## Mapping the Leaf

Information Visualization Final Project

Claire Hattaway | och240

---

### Data Processing

In [1]:
# Retreiving the Word Form Dataset from Lexibank

!wget -O lexibank-analysed-v1.0.zip "https://zenodo.org/records/7836668/files/lexibank%2Flexibank-analysed-v1.0.zip?download=1"

!unzip -q lexibank-analysed-v1.0.zip -d lexibank_v1

--2026-05-03 20:01:01--  https://zenodo.org/records/7836668/files/lexibank%2Flexibank-analysed-v1.0.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.185.48.75, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 33352816 (32M) [application/octet-stream]
Saving to: ‘lexibank-analysed-v1.0.zip’

lexibank-analysed-v 100%[===================>]  31.81M  12.4MB/s    in 2.6s    

2026-05-03 20:01:05 (12.4 MB/s) - ‘lexibank-analysed-v1.0.zip’ saved [33352816/33352816]



In [2]:
from pathlib import Path

root = Path("lexibank_v1")
list(root.rglob("*"))[:20]

[PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/raw'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/metadata.json'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/etc'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/setup.cfg'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/CONTRIBUTORS.md'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/workflow.md'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/.zenodo.json'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/LICENSE'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/TRANSCRIPTION.md'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/test.py'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/cldf'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4c0952/lexibank_lexibank_analysed.py'),
 PosixPath('lexibank_v1/lexibank-lexibank-analysed-a4

In [3]:
from pathlib import Path
import pandas as pd

root = Path("lexibank_v1")

concepts_path = next(root.rglob("concepts.csv"))
languages_path = next(root.rglob("languages.csv"))

forms_zip_matches = list(root.rglob("forms.csv.zip"))
forms_csv_matches = list(root.rglob("forms.csv"))

if forms_zip_matches:
    forms_path = forms_zip_matches[0]
    forms_compression = "zip"
elif forms_csv_matches:
    forms_path = forms_csv_matches[0]
    forms_compression = None
else:
    raise FileNotFoundError("Could not find forms.csv or forms.csv.zip")

print("Concepts:", concepts_path)
print("Languages:", languages_path)
print("Forms:", forms_path)

Concepts: lexibank_v1/lexibank-lexibank-analysed-a4c0952/cldf/concepts.csv
Languages: lexibank_v1/lexibank-lexibank-analysed-a4c0952/cldf/languages.csv
Forms: lexibank_v1/lexibank-lexibank-analysed-a4c0952/cldf/forms.csv.zip


In [4]:
# Filter the dataset for just word forms related to "tea" concept
concepts = pd.read_csv(concepts_path, dtype=str)

tea_concepts = concepts[
    (concepts.get("Concepticon_ID", "").astype(str) == "1373")
    | (concepts.get("Concepticon_Gloss", "").astype(str).str.upper() == "TEA")
    | (concepts.get("Name", "").astype(str).str.upper() == "TEA")
]

tea_concepts

,ID,Name,Description,ColumnSpec,Concepticon_ID,Concepticon_Gloss,Central_Concept,Core_Concept
534,tea,TEA,NaN,NaN,1373,TEA,NaN,NaN


In [5]:
tea_parameter_ids = set(tea_concepts["ID"].astype(str))

# Read the Lexibank dataset in chunks to retrieve the "tea" files
chunks = []

for chunk in pd.read_csv(
    forms_path,
    compression=forms_compression,
    dtype=str,
    chunksize=100_000
):
    tea_chunk = chunk[chunk["Parameter_ID"].astype(str).isin(tea_parameter_ids)]
    chunks.append(tea_chunk)

tea_forms = pd.concat(chunks, ignore_index=True)

tea_forms.head()

,ID,Language_ID,Parameter_ID,Form,Segments,Comment,Source,Value,Local_ID,Graphemes,Profile,Cognacy,Loan,CV_Template,Prosodic_String,Dolgo_Sound_Classes,SCA_Sound_Classes
0,wold-Hawaiian-tea-1,wold-Hawaiian,tea,kī,k i,NaN,WOLD,kī (1),wold-Hawaiian-23-9-1,NaN,NaN,NaN,NaN,CV,CV,KV,KI
1,wold-Takia-tea-1,wold-Takia,tea,ti,t i,NaN,WOLD,ti,wold-Takia-23-9-1,NaN,NaN,NaN,NaN,CV,CV,TV,TI
2,allenbai-Eryuan-tea-1,allenbai-Eryuan,tea,tʂɔ²¹,ʈʂ ɔ ²¹,NaN,Allen2007,tʂɔ²¹,allenbai-Eryuan-205_teadrink-1,NaN,NaN,NaN,NaN,CVT,CVT,KV1,CU3
3,allenbai-Heqing-tea-1,allenbai-Heqing,tea,tsɔu²¹,ts ɔu ²¹,NaN,Allen2007,tsɔu²¹,allenbai-Heqing-205_teadrink-1,NaN,NaN,NaN,NaN,CVT,CVT,KV1,CU3
4,allenbai-Jianchuan-tea-1,allenbai-Jianchuan,tea,tsou²¹,ts ou ²¹,NaN,Allen2007,tsou²¹,allenbai-Jianchuan-205_teadrink-1,NaN,NaN,NaN,NaN,CVT,CVT,KV1,CU3


In [6]:
# Keep only necessary columns
tea_forms_clean = tea_forms[['Language_ID', 'Form', 'Segments']]
tea_forms_clean.head()

,Language_ID,Form,Segments
0,wold-Hawaiian,kī,k i
1,wold-Takia,ti,t i
2,allenbai-Eryuan,tʂɔ²¹,ʈʂ ɔ ²¹
3,allenbai-Heqing,tsɔu²¹,ts ɔu ²¹
4,allenbai-Jianchuan,tsou²¹,ts ou ²¹


In [7]:
# Merge with Language Data

languages = pd.read_csv(languages_path, dtype=str)

tea_full = tea_forms_clean.merge(
    languages,
    left_on="Language_ID",
    right_on="ID",
    how="left",
    suffixes=("", "_language")
)

# Keep only necessary columns
tea_full_clean = tea_full[['Language_ID','Form','Segments','Name','Latitude','Longitude','Family']]
tea_full_clean.head()

,Language_ID,Form,Segments,Name,Latitude,Longitude,Family
0,wold-Hawaiian,kī,k i,Hawaiian,19.5833,-155.5,Austronesian
1,wold-Takia,ti,t i,Takia,-4.66667,146,Austronesian
2,allenbai-Eryuan,tʂɔ²¹,ʈʂ ɔ ²¹,Eryuan,26.538056,99.910000,Sino-Tibetan
3,allenbai-Heqing,tsɔu²¹,ts ɔu ²¹,Heqing,26.0991374,99.9416161,Sino-Tibetan
4,allenbai-Jianchuan,tsou²¹,ts ou ²¹,Jianchuan,26.5536208,100.1396921,Sino-Tibetan


In [8]:
# Separate out languages that have two forms

languages_two_vars = tea_full_clean[tea_full_clean['Name'].duplicated(keep=False)]
languages_two_vars.head()

,Language_ID,Form,Segments,Name,Latitude,Longitude,Family
40,chenhmongmien-NortheastYunnanChuanqiandian,ka³³tɕi⁵⁵dɦu¹¹,k a ³³ + tɕ i ⁵⁵ + dʱ u ¹¹,"Chuanqiandian, Northeast Yunnan",27.096059,103.698607,Hmong-Mien
41,chenhmongmien-NortheastYunnanChuanqiandian,tʂʰa³¹,ʈʂʰ a ³¹,"Chuanqiandian, Northeast Yunnan",27.096059,103.698607,Hmong-Mien
96,wold-Kanuri,tî,t i,Kanuri,12,13,Saharan
97,wold-Kanuri,sháyì,ʃ a j i,Kanuri,12,13,Saharan
116,marrisonnaga-Chokri,cha,tɕʰ a,Chokri,25.6833,94.2667,Sino-Tibetan


In [9]:
# Remove duplicates from original table

tea_full_clean.drop_duplicates(subset='Name', inplace=True)
tea_full_clean.describe()

/tmp/ipykernel_19368/4208818491.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tea_full_clean.drop_duplicates(subset='Name', inplace=True)


,Language_ID,Form,Segments,Name,Latitude,Longitude,Family
count,244,244,244,244,244,244,242
unique,244,197,190,244,240,242,33
top,wold-Hawaiian,t͡ʃaj,tʃ a j,Hawaiian,12,45.50,Sino-Tibetan
freq,1,14,17,1,2,2,86


In [10]:
# Save cleaned files

tea_full_clean.to_csv("lexibank_tea.csv", index=False)
languages_two_vars.to_csv("lexibank_tea_with_vars.csv", index=False)

Process WALS Data for Derivation Classification (World Atlas of Language Structures)

In [11]:
url = 'https://raw.githubusercontent.com/nanxstats/tea-sea-cha-land/refs/heads/master/tea-sea-cha-land.csv'
wals_df = pd.read_csv(url)
wals_df.head()

,id,language,value,description,year,latitude,longitude,genus,family,source,id_url,language_url,source_url
0,abm,Alabama,3,Others,NaN,32.333333,-87.416667,Muskogean,Muskogean,Sylestine et al. n.d.,http://wals.info/valuesets/138A-abm,http://wals.info/languoid/lect/wals_code_abm,http://wals.info/refdb/record/Sylestine-et-al-nd
1,abz,Abaza,1,Words derived from Sinitic cha,NaN,44.000000,42.000000,Northwest Caucasian,Northwest Caucasian,Anonymous 1956,http://wals.info/valuesets/138A-abz,http://wals.info/languoid/lect/wals_code_abz,http://wals.info/refdb/record/Anonymous-1956
2,ace,Acehnese,2,Words derived from Min Nan Chinese te,949.0,5.500000,95.500000,Malayo-Sumbawan,Austronesian,Aboe Bakar et al. 1985,http://wals.info/valuesets/138A-ace,http://wals.info/languoid/lect/wals_code_ace,http://wals.info/refdb/record/Aboe-Bakar-et-al...
3,aeg,Arabic (Egyptian),1,Words derived from Sinitic cha,540.0,30.000000,31.000000,Semitic,Afro-Asiatic,Malherbe and Rosenberg 1996,http://wals.info/valuesets/138A-aeg,http://wals.info/languoid/lect/wals_code_aeg,http://wals.info/refdb/record/Malherbe-and-Ros...
4,afr,Afrikaans,2,Words derived from Min Nan Chinese te,493.0,-31.000000,22.000000,Germanic,Indo-European,Malherbe and Rosenberg 1996,http://wals.info/valuesets/138A-afr,http://wals.info/languoid/lect/wals_code_afr,http://wals.info/refdb/record/Malherbe-and-Ros...


In [27]:
# Drop unnecessary columns

wals_df_clean = wals_df.drop(columns=['source','id_url','language_url','source_url'])
wals_df_clean.head()

,id,language,value,description,year,latitude,longitude,genus,family
0,abm,Alabama,3,Others,NaN,32.333333,-87.416667,Muskogean,Muskogean
1,abz,Abaza,1,Words derived from Sinitic cha,NaN,44.000000,42.000000,Northwest Caucasian,Northwest Caucasian
2,ace,Acehnese,2,Words derived from Min Nan Chinese te,949.0,5.500000,95.500000,Malayo-Sumbawan,Austronesian
3,aeg,Arabic (Egyptian),1,Words derived from Sinitic cha,540.0,30.000000,31.000000,Semitic,Afro-Asiatic
4,afr,Afrikaans,2,Words derived from Min Nan Chinese te,493.0,-31.000000,22.000000,Germanic,Indo-European


Merge Lexibank + WALS Data

In [37]:
merged_df = wals_df_clean.merge(
    tea_full_clean,
    left_on="language",
    right_on="Name",
    how="outer"
)

merged_df.head()

,id,language,value,description,year,latitude,longitude,genus,family,Language_ID,Form,Segments,Name,Latitude,Longitude,Family
0,xoo,!Xóõ,2.0,Words derived from Min Nan Chinese te,NaN,-24.0,21.5,Tu,Tu,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,abz,Abaza,1.0,Words derived from Sinitic cha,NaN,44.0,42.0,Northwest Caucasian,Northwest Caucasian,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,northeuralex-abk,ɑt͡ʃʰɑi,ɑ tʃʰ ɑ i,Abkhaz,43.06,41.16,Abkhaz-Adyge
3,ace,Acehnese,2.0,Words derived from Min Nan Chinese te,949.0,5.5,95.5,Malayo-Sumbawan,Austronesian,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,northeuralex-ady,ɕaːj,ɕ aː j,Adyghe,44.65,40.00,Abkhaz-Adyge


In [38]:
# Move latitude and longitude coordinates to the same column for consistency
# Use WALS coordinates first, then Lexibank coordinates for data not covered by WALS

merged_df['latitude'] = merged_df['latitude'].combine_first(merged_df['Latitude'])
merged_df['longitude'] = merged_df['longitude'].combine_first(merged_df['Longitude'])

# Same procedure for language and family columns
merged_df['language'] = merged_df['language'].combine_first(merged_df['Name'])
merged_df['family'] = merged_df['family'].combine_first(merged_df['Family'])

In [39]:
# Drop duplicate Lexibank columns (Name, Latitude, Longitude, Family) now NaNs have been filled

merged_df = merged_df.drop(columns=['Name', 'Latitude', 'Longitude', 'Family'])

In [40]:
merged_df.head()

,id,language,value,description,year,latitude,longitude,genus,family,Language_ID,Form,Segments
0,xoo,!Xóõ,2.0,Words derived from Min Nan Chinese te,NaN,-24.0,21.5,Tu,Tu,NaN,NaN,NaN
1,abz,Abaza,1.0,Words derived from Sinitic cha,NaN,44.0,42.0,Northwest Caucasian,Northwest Caucasian,NaN,NaN,NaN
2,NaN,Abkhaz,NaN,NaN,NaN,43.06,41.16,NaN,Abkhaz-Adyge,northeuralex-abk,ɑt͡ʃʰɑi,ɑ tʃʰ ɑ i
3,ace,Acehnese,2.0,Words derived from Min Nan Chinese te,949.0,5.5,95.5,Malayo-Sumbawan,Austronesian,NaN,NaN,NaN
4,NaN,Adyghe,NaN,NaN,NaN,44.65,40.00,NaN,Abkhaz-Adyge,northeuralex-ady,ɕaːj,ɕ aː j


In [42]:
# Save cleaned df for reloading if session restarts

from google.colab import drive
drive.mount('/content/drive')

merged_df.to_csv('/content/drive/MyDrive/Mapping_Tea/TeaDerivation+WordForms.csv', index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Reload Data for Session Reconnects

In [1]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/Mapping_Tea/TeaDerivation+WordForms.csv'
df = pd.read_csv(path)

df.head()

Mounted at /content/drive


,id,language,value,description,year,latitude,longitude,genus,family,Language_ID,Form,Segments
0,xoo,!Xóõ,2.0,Words derived from Min Nan Chinese te,NaN,-24.00,21.50,Tu,Tu,NaN,NaN,NaN
1,abz,Abaza,1.0,Words derived from Sinitic cha,NaN,44.00,42.00,Northwest Caucasian,Northwest Caucasian,NaN,NaN,NaN
2,NaN,Abkhaz,NaN,NaN,NaN,43.06,41.16,NaN,Abkhaz-Adyge,northeuralex-abk,ɑt͡ʃʰɑi,ɑ tʃʰ ɑ i
3,ace,Acehnese,2.0,Words derived from Min Nan Chinese te,949.0,5.50,95.50,Malayo-Sumbawan,Austronesian,NaN,NaN,NaN
4,NaN,Adyghe,NaN,NaN,NaN,44.65,40.00,NaN,Abkhaz-Adyge,northeuralex-ady,ɕaːj,ɕ aː j
